In [132]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
import re
from collections import Counter
import warnings
warnings.filterwarnings('ignore')
import os
import nltk
import spacy

In [133]:
path = "C:/Users/maria/PycharmProjects/Urbania_Proyecto/Scrapping"
excel_path = f"{path}/df_limpio.xlsx"
output_path = f"{path}"

In [134]:
df = pd.read_excel(excel_path)

In [135]:
df.head()

,url,distrito_original,direccion,descripcion,fecha_extraccion,otros,tipo_contrato,tipo_vivienda,precio,area_total,area_cubierta,banos,medios_banos,cochera,dormitorio,antiguedad,departamento,provincia,distrito
0,https://urbania.pe/inmueble/clasificado/alclap...,Los Olivos,"LOS CASTAÑOS 1317, 12 De Agosto, Los Olivos",Alquiler Barato Departamento en Los Olivos\nAl...,2025-10-16 19:58:00,"LOS CASTAÑOS 1317, 12 De Agosto",alquiler,departamento,1200,110,110.0,3,0,0,3,5,lima,lima,los olivos
1,https://urbania.pe/inmueble/clasificado/alclap...,Ate Vitarte,"De la Unión 303, 27 De Abril, Ate",Minidepartamento de un Dorm. en Tercer Piso / ...,2025-10-16 19:58:00,"De la Unión 303, 27 De Abril",alquiler,departamento,1300,60,60.0,1,0,0,1,5,lima,lima,ate
2,https://urbania.pe/inmueble/clasificado/alclap...,Ate Vitarte,"De la Unión 303 , 27 De Abril, Ate","Triplex en Tercer, Cuarto y Quinto Piso / Ate ...",2025-10-16 19:59:00,"De la Unión 303, 27 De Abril",alquiler,departamento,2650,160,160.0,1,1,0,3,5,lima,lima,ate
3,https://urbania.pe/inmueble/clasificado/alclap...,Miraflores,"a 2 Cuadras del Óvalo de Higuereta, Piso 2, A...","Alquilo Depa, Urb. Alexander Von Humboldt 2do ...",2025-10-16 19:59:00,"a 2 Cuadras del Óvalo de Higuereta, Piso 2, Al...",alquiler,departamento,3850,144,144.0,2,0,1,3,20,lima,lima,miraflores
4,https://urbania.pe/inmueble/clasificado/alclap...,Cayma,"Cayma Baja, Alto Gloria, Cayma",Departamento Alto de La Gloria - Cayma\nLindo ...,2025-10-16 19:59:00,"Cayma Baja, Alto Gloria",alquiler,departamento,2850,125,110.0,2,1,1,2,8,arequipa,arequipa,cayma


# Nube de palabras (TF-IDF)

In [136]:
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)
STOPWORDS_NLTK = set(stopwords.words('spanish'))

In [137]:
nlp = spacy.load("es_core_news_sm")

Configuración de visualización

In [138]:
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10
sns.set_style("whitegrid")

In [139]:
STOPWORDS_INMOBILIARIAS = {
    'alquiler', 'departamento', 'informacion', 'caracteristicas', 'ubicacion','condominio',
    'precio', 'casa', 'dormitorios', 'baños', 'distrito', 'dormitorio','sala',
    'baño', 'zona', 'cuenta', 'ambiente', 'ambientes',
    'uno','dos','tres', 'av', 'local','comercial',
    'soles','piso','tipo','igv','incluye','ver','datos','alquilo','alquila',
    'cuadras','mes','cuadra'
}


In [140]:
STOPWORDS_ES = STOPWORDS_NLTK.union(STOPWORDS_INMOBILIARIAS)

Limpieza básica de texto

In [141]:
def limpiar_texto(texto):
    texto = str(texto).lower()
    texto = re.sub(r'\d+', '', texto)  # Quitar números
    texto = re.sub(r'[^\wáéíóúüñ\s]', ' ', texto)  # Mantener letras con acentos
    texto = re.sub(r'\s+', ' ', texto)  # Quitar espacios múltiples
    return texto

In [142]:
df_limpio = df.copy()
df_limpio['descripcion_limpia'] = df_limpio['descripcion'].apply(lambda x: limpiar_texto(x))

In [143]:
df_limpio.head()

,url,distrito_original,direccion,descripcion,fecha_extraccion,otros,tipo_contrato,tipo_vivienda,precio,area_total,area_cubierta,banos,medios_banos,cochera,dormitorio,antiguedad,departamento,provincia,distrito,descripcion_limpia
0,https://urbania.pe/inmueble/clasificado/alclap...,Los Olivos,"LOS CASTAÑOS 1317, 12 De Agosto, Los Olivos",Alquiler Barato Departamento en Los Olivos\nAl...,2025-10-16 19:58:00,"LOS CASTAÑOS 1317, 12 De Agosto",alquiler,departamento,1200,110,110.0,3,0,0,3,5,lima,lima,los olivos,alquiler barato departamento en los olivos alq...
1,https://urbania.pe/inmueble/clasificado/alclap...,Ate Vitarte,"De la Unión 303, 27 De Abril, Ate",Minidepartamento de un Dorm. en Tercer Piso / ...,2025-10-16 19:58:00,"De la Unión 303, 27 De Abril",alquiler,departamento,1300,60,60.0,1,0,0,1,5,lima,lima,ate,minidepartamento de un dorm en tercer piso ate...
2,https://urbania.pe/inmueble/clasificado/alclap...,Ate Vitarte,"De la Unión 303 , 27 De Abril, Ate","Triplex en Tercer, Cuarto y Quinto Piso / Ate ...",2025-10-16 19:59:00,"De la Unión 303, 27 De Abril",alquiler,departamento,2650,160,160.0,1,1,0,3,5,lima,lima,ate,triplex en tercer cuarto y quinto piso ate cer...
3,https://urbania.pe/inmueble/clasificado/alclap...,Miraflores,"a 2 Cuadras del Óvalo de Higuereta, Piso 2, A...","Alquilo Depa, Urb. Alexander Von Humboldt 2do ...",2025-10-16 19:59:00,"a 2 Cuadras del Óvalo de Higuereta, Piso 2, Al...",alquiler,departamento,3850,144,144.0,2,0,1,3,20,lima,lima,miraflores,alquilo depa urb alexander von humboldt do pis...
4,https://urbania.pe/inmueble/clasificado/alclap...,Cayma,"Cayma Baja, Alto Gloria, Cayma",Departamento Alto de La Gloria - Cayma\nLindo ...,2025-10-16 19:59:00,"Cayma Baja, Alto Gloria",alquiler,departamento,2850,125,110.0,2,1,1,2,8,arequipa,arequipa,cayma,departamento alto de la gloria cayma lindo dep...


Normalización

In [144]:
def lematizar_texto(texto):

    doc = nlp(texto)
    tokens = [token.lemma_ for token in doc
              if not token.is_stop and len(token.text) > 2 and token.text not in STOPWORDS_ES]

    return ' '.join(tokens)

In [145]:
df_limpio['descripcion_lematizada'] = df_limpio['descripcion_limpia'].apply(lematizar_texto)

In [146]:
df_limpio.head()

,url,distrito_original,direccion,descripcion,fecha_extraccion,otros,tipo_contrato,tipo_vivienda,precio,area_total,...,banos,medios_banos,cochera,dormitorio,antiguedad,departamento,provincia,distrito,descripcion_limpia,descripcion_lematizada
0,https://urbania.pe/inmueble/clasificado/alclap...,Los Olivos,"LOS CASTAÑOS 1317, 12 De Agosto, Los Olivos",Alquiler Barato Departamento en Los Olivos\nAl...,2025-10-16 19:58:00,"LOS CASTAÑOS 1317, 12 De Agosto",alquiler,departamento,1200,110,...,3,0,0,3,5,lima,lima,los olivos,alquiler barato departamento en los olivos alq...,barato olivo olivo ubicación castaño olivo res...
1,https://urbania.pe/inmueble/clasificado/alclap...,Ate Vitarte,"De la Unión 303, 27 De Abril, Ate",Minidepartamento de un Dorm. en Tercer Piso / ...,2025-10-16 19:58:00,"De la Unión 303, 27 De Abril",alquiler,departamento,1300,60,...,1,0,0,1,5,lima,lima,ate,minidepartamento de un dorm en tercer piso ate...,minidepartamento dorm tercer ate cerca ovalo s...
2,https://urbania.pe/inmueble/clasificado/alclap...,Ate Vitarte,"De la Unión 303 , 27 De Abril, Ate","Triplex en Tercer, Cuarto y Quinto Piso / Ate ...",2025-10-16 19:59:00,"De la Unión 303, 27 De Abril",alquiler,departamento,2650,160,...,1,1,0,3,5,lima,lima,ate,triplex en tercer cuarto y quinto piso ate cer...,triplex tercer cuarto quinto ate cerca ovalo s...
3,https://urbania.pe/inmueble/clasificado/alclap...,Miraflores,"a 2 Cuadras del Óvalo de Higuereta, Piso 2, A...","Alquilo Depa, Urb. Alexander Von Humboldt 2do ...",2025-10-16 19:59:00,"a 2 Cuadras del Óvalo de Higuereta, Piso 2, Al...",alquiler,departamento,3850,144,...,2,0,1,3,20,lima,lima,miraflores,alquilo depa urb alexander von humboldt do pis...,depa urb alexander von humboldt miraflor cuadr...
4,https://urbania.pe/inmueble/clasificado/alclap...,Cayma,"Cayma Baja, Alto Gloria, Cayma",Departamento Alto de La Gloria - Cayma\nLindo ...,2025-10-16 19:59:00,"Cayma Baja, Alto Gloria",alquiler,departamento,2850,125,...,2,1,1,2,8,arequipa,arequipa,cayma,departamento alto de la gloria cayma lindo dep...,alto gloria caymar lindo mtr totalmente amobla...


## Análisis TF-IDF por distrito

In [147]:
def analisis_tfidf(df, top_n = 20):
    df_distrito = df.groupby('distrito')['descripcion_limpia'].apply(lambda x: ' '.join(x)).reset_index()

    vectorizer = TfidfVectorizer(
        max_features=200,
        stop_words=list(STOPWORDS_ES),
        ngram_range=(1, 2),
        min_df=1
    )

    tfidf_matrix = vectorizer.fit_transform(df_distrito['descripcion_limpia'])
    feature_names = vectorizer.get_feature_names_out()

    resultados = {}

    for idx, distrito in enumerate(df_distrito['distrito']):
        tfidf_scores = tfidf_matrix[idx].toarray()[0]
        top_indices = tfidf_scores.argsort()[-top_n:][::-1]
        top_words = [(feature_names[i], tfidf_scores[i]) for i in top_indices if tfidf_scores[i] > 0]
        resultados[distrito] = top_words

    return resultados

In [148]:
resultados_tfidf = analisis_tfidf(df_limpio, top_n=20)

In [149]:
print(resultados_tfidf)

{'asia': [('incorporado', 0.3216742688869255), ('mar', 0.30123228366197174), ('nivel', 0.2952555207570015), ('terraza', 0.2696952483687522), ('piscina', 0.25274406841115576), ('servicio', 0.20745788336057863), ('balcón', 0.17655255368119077), ('áreas', 0.17447116982090694), ('cama', 0.16677137514805362), ('niños', 0.16375125923337555), ('área', 0.16005912216277124), ('principal', 0.15298819067861888), ('estacionamientos', 0.15010687089456914), ('parrilla', 0.14842502617738648), ('vista', 0.1421146904904981), ('cuarto', 0.13544726811447452), ('equipada', 0.1308533773656802), ('comedor', 0.12739810392238116), ('completo', 0.12621659080667194), ('santa', 0.11287272342872878)], 'ate': [('industrial', 0.4226586052966677), ('m²', 0.33229824546375436), ('oficinas', 0.19660657680356217), ('santa', 0.19426657427176686), ('nivel', 0.18973796075783012), ('ideal', 0.14746245595880542), ('negocio', 0.13795430745354068), ('ubicación', 0.13472962382254325), ('frente', 0.11968626256174911), ('lima', 0

## Nube de palabras

In [150]:
def generar_nubes_palabras(resultados_tfidf, output_folder='Nube palabras distrito'):

    os.makedirs(output_path, exist_ok=True)

    for distrito, palabras in resultados_tfidf.items():
        if not palabras:
            continue

        # Diccionario
        word_dict = {palabra: score for palabra, score in palabras}

        # Nube
        wordcloud = WordCloud(
            width=800,
            height=400,
            background_color='white',
            colormap='viridis',
            relative_scaling=0.5,
            min_font_size=10
        ).generate_from_frequencies(word_dict)

        plt.figure(figsize=(12, 6))
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.title(f'Palabras Clave - {distrito}', fontsize=16, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()

        filename = f"{output_path}/{output_folder}/nube_{distrito.replace(' ', '_')}.png"
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        plt.close()

    print(f"✓ {len(resultados_tfidf)} nubes de palabras guardadas")

In [151]:
generar_nubes_palabras(resultados_tfidf)

✓ 73 nubes de palabras guardadas
